# ¿Avanzan los gobiernos autoritarios? Freedom in the World 2013-2020

Cómo cambió entre 2013 y 2020 la clasificación de los países en el informe *Freedom in the World* de Freedom House:

1. Número de países libres, parcialmente libres y no libres en cada edición.
2. Mapas de la variación de la puntuación en derechos políticos (`pr`) y libertades civiles (`cl`).

**Datos necesarios** (ver [`data/README.md`](../data/README.md)): `staging/mundial/freedom_house.xlsx`, `staging/mundial/gini_mundial.xls` (solo se usa para obtener el código de país del Banco Mundial, que es el que usa el mapa) y `raw/mapas/countries.geojson`.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px

sys.path.insert(0, str(Path.cwd().parent))  # permite importar el paquete politicai desde notebooks/
from politicai.config import RAW_DIR, STAGING_DIR
from politicai.limpieza import clean_country_names, clean_df_format

pd.set_option("display.max_columns", None)

MUNDIAL_DIR = STAGING_DIR / "mundial"
GEOJSON_PATH = RAW_DIR / "mapas" / "countries.geojson"

## Datos

In [ ]:
df_freedomhouse = clean_df_format(pd.read_excel(MUNDIAL_DIR / "freedom_house.xlsx"))
df_gini = clean_df_format(pd.read_excel(MUNDIAL_DIR / "gini_mundial.xls"))

In [ ]:
# Ediciones que se comparan, con el código de país del Banco Mundial para el mapa.
df_base = clean_country_names(df_freedomhouse, column="country_territory")
df_base = df_base[df_base["edition"].isin([2013, 2020])]

codigos_wb = df_gini[["country_name", "country_code"]].drop_duplicates().rename(columns={"country_code": "code_wb"})
codigos_wb = clean_country_names(codigos_wb, column="country_name")
df_base = df_base.merge(codigos_wb, on="country", how="inner")
print("países:", df_base["country"].nunique(), "| filas:", df_base.shape)

## Evolución del número de países por estatus

Se usan todas las ediciones del informe, no solo 2013 y 2020.

In [ ]:
df_count = df_freedomhouse.groupby(["edition", "status"]).size().reset_index(name="count")
df_count["status"] = df_count["status"].replace({"F": "Libre", "NF": "No libre", "PF": "Parcialmente libre"})
df_count

In [ ]:
fig_estatus = px.line(
    df_count, x="edition", y="count", color="status",
    title="Evolución del estatus de los países 2013-2020 según Freedom in the World",
)
fig_estatus.show()

## Variación 2013-2020 por país

`pr` (derechos políticos) y `cl` (libertades civiles) son las puntuaciones del informe; una variación negativa indica un retroceso.

In [ ]:
columnas = ["country", "region", "edition", "status", "pr", "cl", "code_wb"]
df_2013 = df_base.loc[df_base["edition"] == 2013, columnas].rename(
    columns={"status": "status_13", "pr": "pr_13", "cl": "cl_13"}
)
df_2020 = df_base.loc[df_base["edition"] == 2020, ["code_wb", "pr", "cl", "status"]].rename(
    columns={"status": "status_20", "pr": "pr_20", "cl": "cl_20"}
)
df_variation = df_2013.merge(df_2020, on="code_wb", how="left")
df_variation["pr_var"] = df_variation["pr_20"] - df_variation["pr_13"]
df_variation["cl_var"] = df_variation["cl_20"] - df_variation["cl_13"]
df_variation.head()

In [ ]:
with open(GEOJSON_PATH) as geofile:
    geojson = json.load(geofile)


def mapa_variacion(columna, titulo):
    """Mapa mundial de la variación de una puntuación, centrado en 0 (rojo = retroceso)."""
    return px.choropleth_map(
        data_frame=df_variation, geojson=geojson, locations="code_wb", featureidkey="id",
        color=columna, color_continuous_midpoint=0, color_continuous_scale="Picnic_r",
        hover_data=["country", "status_20"],
        map_style="carto-positron", zoom=0.5, center={"lat": 20, "lon": 0},
        title=titulo,
    )

In [ ]:
fig_derechos = mapa_variacion("pr_var", "Variación de puntuación en Derechos Políticos 2013-2020 por Freedom in the World")
fig_derechos.show()

In [ ]:
fig_libertades = mapa_variacion("cl_var", "Variación de puntuación en Libertades Civiles 2013-2020 por Freedom in the World")
fig_libertades.show()